# M1M3 z-Gradient Survey (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-07-07
**Last Modified:** 2026-07-07
**Status:** In Progress
**Keywords:** M1M3, thermal gradient, z_gradient, EFD, thermocouples, bounce test

## Description

Survey the M1M3 vertical thermal gradient (`z_gradient`) across all `science` and
`acq` exposures over a date range.  `z_gradient` is **not** in the ConsDB — it is
derived from the EFD M1M3 thermocouple telemetry via
`lsst.ts.m1m3.utils.ThermocoupleAnalysis`, the same computation the AOS
measured-intrinsic pipeline (`ts_intrinsic_wavefront`) uses to enrich its visit
tables.  This notebook pulls the exposure list from the Butler, computes the
gradient per exposure night-by-night (with a per-day progress bar), caches the
result to parquet, and plots the trend.

Key functionality:
1. List `science`/`acq` exposures per night from the Butler over [START, END].
2. Compute `z_gradient` per exposure from the EFD (`get_m1m3_data`), one night at a time.
3. Cache to a small parquet and plot z_gradient vs ordinal day_obs (scatter + nightly median)
   and its per-night distribution (violin).

**Output:** `z_gradient_science_acq.parquet` + a scatter/violin figure.

**References:** `ts_intrinsic_wavefront/python/lsst/ts/intrinsic/wavefront/intrinsics_lib.py`
(`get_m1m3_data`); ConsDB schema `cdb_lsstcam` (confirms z_gradient is not a ConsDB column).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-07 | Aaron Roodman | Initial version — science/acq z_gradient survey (fetch + scatter/violin) |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Fetch z_gradient per exposure](#fetch)
4. [Plots](#plots)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — all configurable values collected here
# ============================================================
from datetime import date

START = "2026-03-15"                 # first calendar day (inclusive)
END = date.today().isoformat()       # last calendar day (inclusive) — default: today
KEEP_TYPES = ("science", "acq")      # exposure.observation_type values to keep
BUTLER_REPO = "/repo/main"
EFD_NAME = "usdf_efd"                 # EFD instance for this site
OUT = "z_gradient_science_acq.parquet"   # cache path (cwd); set elsewhere if desired
FIG = "z_gradient_science_acq.png"

<a id='setup'></a>
## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from lsst.daf.butler import Butler
from lsst_efd_client import EfdClient
from lsst.ts.intrinsic.wavefront.intrinsics_lib import get_m1m3_data

butler = Butler(BUTLER_REPO, instrument="LSSTCam")
# NOTE: construct EfdClient directly with output_mode="dataframe"; makeEfdClient()
# can fail with "Invalid output format" on some summit_utils/lsst_efd_client combos.
efd = EfdClient(EFD_NAME, output_mode="dataframe")

<a id='fetch'></a>
## Fetch z_gradient per exposure

One night at a time; `await` works at notebook top level. In a plain script wrap the loop in `async def` and use `asyncio.run(...)`.

In [ ]:
# One EFD ThermocoupleAnalysis load per night (skips empty nights); a few minutes
# over a multi-month range.  Per-day failures are logged and set to NaN, not fatal.
days = [int(d.strftime("%Y%m%d")) for d in pd.date_range(START, END)]

rows = []
for day in tqdm(days, desc="day_obs"):
    recs = butler.registry.queryDimensionRecords(
        "exposure", instrument="LSSTCam", where=f"exposure.day_obs = {day}")
    sci = [r for r in recs if r.observation_type in KEEP_TYPES]
    if not sci:
        continue
    vt = pd.DataFrame({
        "day_obs": day,
        "seq_num": [r.seq_num for r in sci],
        "observation_type": [r.observation_type for r in sci],
        "obs_start": [r.timespan.begin.tai.isot for r in sci],
    })
    try:
        grad = await get_m1m3_data(efd, vt, include_thermocouples=False,
                                   include_cell_gradients=False)
        vt["z_gradient"] = np.asarray(grad["z_gradient"], float)
    except Exception as e:
        tqdm.write(f"  {day}: gradient failed ({type(e).__name__}: {e})")
        vt["z_gradient"] = np.nan
    rows.append(vt)

df = pd.concat(rows, ignore_index=True)
df.to_parquet(OUT)
print(f"{len(df)} science/acq exposures over {df.day_obs.nunique()} nights -> {OUT}")

<a id='plots'></a>
## Plots

In [ ]:
df = pd.read_parquet(OUT)
df = df[np.isfinite(df.z_gradient)]
days = np.sort(df.day_obs.unique())
ordmap = {d: i for i, d in enumerate(days)}          # ordinal night index (gap-free)
df["day_ord"] = df.day_obs.map(ordmap)
ticks = np.arange(0, len(days), max(1, len(days) // 15))

fig, (a1, a2) = plt.subplots(2, 1, figsize=(max(10, 0.22 * len(days)), 9),
                             constrained_layout=True)

# scatter: one point per exposure + nightly median
a1.scatter(df.day_ord, df.z_gradient, s=6, alpha=0.25, color="steelblue")
med = df.groupby("day_ord").z_gradient.median()
a1.plot(med.index, med.values, "-o", ms=3, color="crimson", lw=0.8, label="nightly median")
a1.set_ylabel("M1M3 z_gradient"); a1.legend(fontsize=8); a1.grid(alpha=0.3)
a1.set_title(f"z_gradient per science/acq exposure  "
             f"({df.day_obs.min()}-{df.day_obs.max()}, N={len(df)})")
a1.set_xticks(ticks); a1.set_xticklabels([str(days[i]) for i in ticks], rotation=90, fontsize=7)

# violin: distribution per night
data = []
for i in range(len(days)):
    d = df.z_gradient[df.day_ord == i].values
    data.append(d if d.size >= 2 else (np.full(2, d[0]) if d.size == 1
                                       else np.array([np.nan, np.nan])))
vp = a2.violinplot(data, positions=np.arange(len(days)), widths=0.9,
                   showmedians=True, showextrema=False)
for b in vp["bodies"]:
    b.set_facecolor("steelblue"); b.set_alpha(0.5)
a2.set_ylabel("M1M3 z_gradient"); a2.set_xlabel("day_obs"); a2.grid(axis="y", alpha=0.3)
a2.set_xticks(ticks); a2.set_xticklabels([str(days[i]) for i in ticks], rotation=90, fontsize=7)

fig.savefig(FIG, dpi=140)
plt.show()